# Lab 06: Finding Nearest Neighbors in Sparse Spaces

**Lab Objective:** In high-dimensional data, especially text, vectors are extremely sparse. This lab teaches you to leverage that sparsity for efficiency. We will move beyond dense computations to manipulate **Compressed Sparse Row (CSR)** matrices and implement advanced pruning techniques.

**Key Learning Points:**
1. Manipulating and traversing `scipy.sparse.csr_matrix` data structures.
2. Implementing an inverted index and score accumulation directly from sparse pointers.
3. Understanding $L^2$-norm pre-processing and suffix-based pruning bounds.
4. Analyzing the precision-recall trade-offs in LSH hashing families.

## 1. Setup: Sparse Data and Pre-processing

For Cosine Similarity, we are interested in the dot product of unit vectors: 
$$sim(x,y) = \sum_{j=1}^{m} x_j \times y_j$$
This is only true if $||x||_2 = 1$ and $||y||_2 = 1$. Instead of normalizing during every dot product, we normalize the entire dataset once as a pre-processing step.

In [ ]:
import numpy as np
import time
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from lsh import clsh, recall

# Load data
categories = ['sci.med', 'sci.space']
data = fetch_20newsgroups(subset='train', categories=categories)

# Vectorize (Returns a csr_matrix)
vectorizer = TfidfVectorizer(max_features=2000, stop_words='english')
X_sparse = vectorizer.fit_transform(data.data)

# Pre-processing: Normalize each row to unit length (L2 norm)
X = normalize(X_sparse, norm='l2', axis=1)

print(f"Dataset Shape: {X.shape}")
print(f"Matrix Type: {type(X)}")

### 1.1 Understanding the CSR Matrix
The `csr_matrix` does not store zeros. It uses three arrays:
- `X.data`: The non-zero values.
- `X.indices`: The column (feature) IDs for those values.
- `X.indptr`: Pointers to the start/end of each row. 

To access non-zeros for row `i`, you look at `X.data[X.indptr[i] : X.indptr[i+1]]`.

## 2. Problem 1: CSR-Based IdxJoin

### 2.1 Task: Building the Inverted Index
**Task:** Build an inverted index by traversing the `csr_matrix` pointers. The index should map `feature_id` to a list of `(doc_id, value)`.

In [ ]:
def build_inverted_index_sparse(X):
    inv_idx = {}
    num_docs = X.shape[0]
    
    #### YOUR CODE HERE: Use X.indptr, X.indices, and X.data ####
    
    return inv_idx
    ####

inverted_index = build_inverted_index_sparse(X)
print(f"Built index for {len(inverted_index)} features.")

### 2.2 Task: Accumulator Query
**Task:** Implement the IdxJoin query. For a query document, you must iterate through its non-zero features and update the accumulator using the inverted index.

In [ ]:
def sparse_idx_join_query(doc_id, X, inv_idx):
    num_docs = X.shape[0]
    accumulator = np.zeros(num_docs)
    
    # Get query features and values using CSR pointers
    start, end = X.indptr[doc_id], X.indptr[doc_id+1]
    q_feats = X.indices[start:end]
    q_vals = X.data[start:end]
    
    #### YOUR CODE HERE: Logic to populate accumulator ####

    ####
    
    accumulator[doc_id] = 0 # Ignore self
    return accumulator

sims = sparse_idx_join_query(0, X, inverted_index)

## 3. Problem 2: The Pruning Point

As vectors are unit normalized, the dot product of a suffix (the remaining features) can be upper-bounded using the Cauchy-Schwarz inequality:
$$dot(x', y') \le ||x'||_2 \times ||y'||_2$$
If the current accumulated similarity plus this upper bound is less than our threshold $\epsilon$, the candidate can be pruned immediately.

**Task:** For two sparse vectors (Row 0 and Row 10 of X), calculate the dot product feature-by-feature. At each step, calculate the **remaining $L^2$-norm** of both vectors. Identify the first feature index where the pair could be pruned for a threshold of **0.8**.

**Note:** Efficiently calculating this 'remaining norm' is the core of the L2AP algorithm. For more details, see: https://davidanastasiu.net/pdf/papers/2014-AnastasiuK-ICDE-l2ap.pdf

In [ ]:
def get_remaining_norm(vector_data, start_idx):
    """Calculates sqrt of sum of squares of values from start_idx onwards"""
    return np.sqrt(np.sum(np.square(vector_data[start_idx:])))

def find_pruning_point(doc_a_id, doc_b_id, X, threshold=0.8):
    # Extract data for both vectors
    # Hint: Focus only on features that both vectors actually share
    
    #### YOUR CODE HERE ####
    
    print(f"Pruned at step X because bound Y < {threshold}")
    ####
    pass

## 4. Problem 3: Approximate Search with LSH

LSH partitions vectors into 'buckets'. If two vectors are similar, they have a high probability of colliding (landing in the same bucket). In this lab, we will observe the give and take between building multiple hash tables and employing multiple hash functions when searching for approximate neighbors using LSH. 

I have written a basic LSH implementation in Python, with instantiations for the cosine similarity, Hamming distance, and the Jaccard coefficient LSH families. The code is written in OOP style and can be easily extended to other LSH families. Open the lsh.py file and study/read the code for details. In the example below, we will be using the cosine similarity version of the LSH data structure, *clsh*.



In [ ]:
import time
import numpy as np
from lsh import clsh, jlsh, generateSamples, findNeighborsBrute, recall

First, we will generate some random samples, and split the data into train (X) and test (Y) subsets. Samples are generated from 100 gausian blobs, i.e., points will be fairly spread out as far as their cosine similarity is concerned.

In [ ]:
X, Y = generateSamples(nsamples=1000, nfeatures=100, nclusters=64, clusterstd=50, binary=False)
print(X.shape, Y.shape)

The basic concept in LSH is that of *hashing* the vectors using a random LSH family of hash functions. As we discussed in class, the LSH families will be more likely to assign the same hash value to similar items. This, however, does not happen all the time. First, let's see what the result of hashing a vector looks like.

In [ ]:
L11 = clsh(X, ntables=1, nfunctions=1)
for i in range(10):
    print(L11.hash(X[i,:]))

As shown in the slides, the output of the cosine family of LSH function is binary, depending on the sign of the dot-product $\langle r,x\rangle$ between the random unit vector $r$ and our input vector $x=X[i,:]$.

Note that we created a single table in our LSH data structure and are using a single LSH function to hash vectors. This means that we're simply partitioning vectors into two buckets. Some vectors will go to the bucket with ID 0, and others will go to the bucket with ID 1.

When we instantiated the LSH data structure *L*, all the vectors in X were already assigned to their respective buckets. Let's see how many vectors each bucket has.

In [ ]:
print("Bucket ID 0 has %d vectors." % len(L11.tables[0]['0']))
print("Bucket ID 1 has %d vectors." % len(L11.tables[0]['1']))

Now, when it's time to find neighbors for a new vector, say $y=Y[0,:]$, the first vector in our test set, we hash the vector to see which bucket we should look in to find neighbors.

In [ ]:
print(L11.hash(Y[0,:], tid=0, fid=0))

Note that I passed in the ID of the table I'm searching in and the ID of the function I'm hashing with. For LSH to work, we have to use the same hashing functions that were used to create the table(s). Therefore, $clsh$ stores the randomly generated functions it created for each table.

Now, it looks like I have to compare $y$ against almost half of the vectors in $X$, which is a lot, and leads to low *precision*. Precision is the fraction of retrieved instances (the vectors we compared against) that are relevant (that would also be in the exact result). Since the number of objects we're comparing against is high, precision will be low. In order to increase the precision, I can use several hash functions and concatenate their results. Increasing the precision will also reduc the amount of time spent finding neighbors, as we will have fewer objects to compare against.

Let's say I use 2 hash functions from the Cosine LSH family. Then, the possible resulting hash values would be 00, 01, 10, and 11, spliting the vectors in X into 4 buckets (instead of 2, when we used 1 function). If we use 3 functions, we get 8 buckets. In general, using $f$ functions will split the "search space" into $2^f$ buckets.

Let's try this using 3 functions.

In [ ]:
L13 = clsh(X, ntables=1, nfunctions=3)
for k in L13.tables[0].keys():
    print("Bucket ID %s has %d vectors." % (k, len(L13.tables[0][k])))
    
print("\nWe only need to compare y against vectors in bucket %s." % L13.signature(Y[0,:], tid=0))

**Side note**: Note that in this academic LSH implementation we use a simple way to generate bucket IDs. We concatenate the string representation of the resulting hash value from each hash function. LSH libraries often implement a secondary (exact) hash function for generating numeric IDs for the buckets. A similar scheme is proposed in the LSH reference I nored on Canvas: [SPM'08] Malcolm Slaney and Michael Casey. Locality-Sensitive Hashing for Finding Nearest Neighbors. Lecture Notes. IEEE Signal Processing Magazine, 2008.

It is easy to see we now have much fewer vectors to compare against when we search for $y$'s neighbors. However, some of the true neighbors may have been accidentally placed in other buckets, which lowers *recall*. Recall (also known in Statistics references as *sensitivity*) is the fraction of relevant instances that are retrieved, i.e., the fraction of true neighbors in our top-$k$ divided by $k$. 

Let's compare the mean recall for finding neighbors using 1 hash function vs. 3 hash functions. To do that, we will first have to find the "true neighbors".

In [ ]:
k = 100  # number of neighbors to find
nbrsExact = findNeighborsBrute(X, Y, k=k, sim="cos")
print("Number of computed similarities for the brute-force approach: %d." % (X.shape[0] * Y.shape[0]))
nbrsTest11  = L11.findNeighbors(Y, k=k)
nbrsTest13  = L13.findNeighbors(Y, k=k)
print("Recall with 1 hash function: %f. Number of computed similarities: %d." % (recall(nbrsTest11, nbrsExact), L11.nsims))
print("Recall with 3 hash functions: %f. Number of computed similarities: %d." % (recall(nbrsTest13, nbrsExact), L13.nsims))

We can increase the recall by building several LSH tables instead of one. Then, instead of looking in one bucket for $y$'s neighbors, we will be looking in one bucket in each table. The search method gets the set union of object IDs in all these buckets, and then computes similarities against all of them.

### 4.1 Precision and Recall in LSH
Using $f$ functions in a single table splits the search space into $2^f$ buckets. 
- **AND Operation:** Using multiple functions (increasing $f$) increases **Precision** by making buckets more specific, but lowers **Recall** because true neighbors might miss a match on just one function.
- **OR Operation:** Using multiple tables (increasing $t$) increases **Recall** by giving neighbors more chances to collide, but lowers **Precision** by increasing the number of candidates to check.

**Task:** Compare the mean recall for finding neighbors using 1 table vs. 3 tables, when each table uses 3 hash functions.

In [ ]:
L33 = 
nbrsTest33  = ...
print("Recall with 3 tables and 3 hash functions: %f. Number of computed similarities: %d." % (recall(nbrsTest33, nbrsExact), L33.nsims))

Given high enough # tables and # hashes (hash functions), we can achieve high recall and precision, sometimes at the expense of efficiency.

**Task:** Find the minimum number of tables (`ntables`) needed to achieve a `recall` of at least **0.90** using `nfunctions=3` for the provided dataset. What is the number of computed similarities for that LSH forest?

In [ ]:
# your code here


Repeat the two tasks above using `Jaccard Coefficient` instead of `cosine similarity`. Note that you will need to re-generate samples using the `binary=True` parameter and re-compute `nbrsExact` for the new similarity measure.

In [ ]:
# your code here


## 5. Reflection Questions

1. **Pre-processing:** Why does normalizing the vectors to unit length at the beginning save time during the search process?
2. **CSR Traversal:** How does the `indptr` array help us skip documents that don't share any features with our query?
3. **Pruning Logic:** Based on the Cauchy-Schwarz bound $dot(x', y') \le ||x'||_2 \times ||y'||_2$, how would sorting features by their weight (value) improve pruning speed?
4. **LSH Trade-offs:** In Exercise 4, as you increased the number of tables to reach 0.90 recall, what happened to the 'Number of computed similarities'? Is LSH always faster than exact search?